In [27]:
import os
import json
import time
import pathlib
import re
from string import Template
from dotenv import load_dotenv
from openai import OpenAI
from pymed import PubMed
from typing import List, Dict
import numpy as np
import torch
import faiss
from transformers import AutoTokenizer, AutoModel


In [28]:
BASE_DIR = pathlib.Path.cwd()

load_dotenv(BASE_DIR / "environment.env")

OPENAI_API_KEY = os.getenv("OPENAI_API_KEY")
if not OPENAI_API_KEY:
    raise RuntimeError("OPENAI_API_KEY not set in .env")



In [29]:
with open(BASE_DIR / "config.json", "r", encoding="utf-8") as f:
    CONFIG = json.load(f)

PUBMED_CFG = CONFIG["pubmed"]
OPENAI_CFG = CONFIG["openai"]
OUTPUT_CFG = CONFIG["output"]

# Optional manual PubMed IDs and claim override to bypass search
MANUAL_PUBMED_IDS = PUBMED_CFG.get("manual_pubmed_ids", [])
MANUAL_CLAIM = CONFIG.get("manual_claim", {})


In [30]:
RETRIEVAL_RESULTS_PATH = pathlib.Path(CONFIG.get("retrieval", {}).get("results_path", BASE_DIR / "retrieval_outputs/retrieval_results.jsonl"))

def load_retrieval_results(path: pathlib.Path) -> Dict[str, List[Dict]]:
    if not path.exists():
        raise FileNotFoundError(f"Missing retrieval results: {path}. Run retrieval_pubmed_faiss.ipynb first.")
    by_claim = {}
    with path.open("r", encoding="utf-8") as f:
        for line in f:
            line = line.strip()
            if not line:
                continue
            rec = json.loads(line)
            by_claim[rec.get("claim_id")] = rec.get("articles", [])
    return by_claim

RETRIEVAL_BY_CLAIM = load_retrieval_results(RETRIEVAL_RESULTS_PATH)


In [31]:
OUTPUT_DIR = BASE_DIR / OUTPUT_CFG.get("output_dir")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

In [32]:
with open(BASE_DIR / "claims.json", "r", encoding="utf-8") as f:
    CLAIMS = json.load(f)

In [33]:
with open(BASE_DIR / "toulmin_prompt.txt", "r", encoding="utf-8") as f:
    TOULMIN_TEMPLATE = Template(f.read())

In [34]:
client = OpenAI(api_key=OPENAI_API_KEY)

pubmed = PubMed(
    tool=PUBMED_CFG["tool"],
    email=PUBMED_CFG["email"]
)

In [35]:
def build_rct_query(drug: str, condition: str, outcome: str) -> str:
    parts = [
        f'"{drug}"[Title/Abstract]'
    ]
    if condition:
        parts.append(f'"{condition}"[Title/Abstract]')
    if outcome:
        parts.append(f'"{outcome}"[Title/Abstract]')
    parts.append('"randomized controlled trial"[Publication Type]')
    return " AND ".join(parts)


In [36]:
def build_review_query(drug: str, condition: str, outcome: str) -> str:
    parts = [
        f'"{drug}"[Title/Abstract]'
    ]
    if condition:
        parts.append(f'"{condition}"[Title/Abstract]')
    if outcome:
        parts.append(f'"{outcome}"[Title/Abstract]')
    parts.append('("systematic review"[Title/Abstract] OR "meta-analysis"[Title/Abstract])')
    return " AND ".join(parts)


In [37]:
def normalize_year(year_value) -> int:
    """Coerce a variety of 'year' representations to an int year; return 0 if unknown."""
    if isinstance(year_value, int):
        return year_value
    if isinstance(year_value, str):
        try:
            return int(year_value)
        except Exception:
            import re
            m = re.search(r'\d{4}', year_value)
            if m:
                return int(m.group(0))
            return 0
    # datetime/date-like objects
    try:
        y = getattr(year_value, "year", None)
        if isinstance(y, int):
            return y
    except Exception:
        pass
    return 0


def filter_by_year(articles: List[Dict], year_from: int) -> List[Dict]:
    return [a for a in articles if normalize_year(a.get("year")) >= year_from]


In [38]:
def fetch_articles(query: str, max_results: int) -> List[Dict]:
    results = pubmed.query(query, max_results=max_results)
    articles = []
    sleep_s = PUBMED_CFG.get("sleep_between_requests", 0.34)

    for article in results:
        art = article.toDict()
        pub_date = art.get("publication_date")
        year = None
        if pub_date is not None:
            try:
                # pub_date may be a date-like object or string
                year = getattr(pub_date, "year", pub_date)
            except Exception:
                pass

        # Normalize year to an int (0 if unknown)
        try:
            art["year"] = normalize_year(year)
        except Exception:
            art["year"] = 0

        # Ensure publication_types is a list for downstream checks
        pts = art.get("publication_types")
        if pts is None:
            art["publication_types"] = []
        else:
            art["publication_types"] = pts

        # Abstract normalisation
        if isinstance(art.get("abstract"), list):
            art["abstract"] = " ".join(art["abstract"])

        articles.append(art)
        time.sleep(sleep_s)

    return articles


In [39]:
def fetch_articles_by_ids(pubmed_ids):
    """Fetch articles directly by PubMed ID using the existing fetch_articles helper."""
    articles = []
    ids = pubmed_ids or []
    for pid in ids:
        q = f"{pid}[PMID]"
        fetched = fetch_articles(q, max_results=1)
        if fetched:
            articles.append(fetched[0])
    return articles


In [40]:
def is_rct(art: Dict) -> bool:
    """
    Detect RCTs robustly:
    - check publication_types if present
    - fallback to scanning title/abstract/methods for trial-related keywords
    """
    # Normalize publication types
    pts = []
    raw_pts = art.get("publication_types")
    if raw_pts:
        # sometimes publication_types is a list of dicts/strings; coerce to strings
        for pt in raw_pts:
            if isinstance(pt, str):
                pts.append(pt.lower())
            else:
                try:
                    pts.append(str(pt).lower())
                except Exception:
                    pass

    # Strong signal from publication_types
    for pt in pts:
        if any(sig in pt for sig in ("randomized controlled trial", "randomised controlled trial",
                                     "randomized", "randomised", "randomised trial", "randomized trial")):
            return True

    # Fallback: look for keywords in title/abstract/methods/etc.
    title = (art.get("title") or "").lower()
    abstract = (art.get("abstract") or "").lower()
    methods = (art.get("methods") or "").lower()
    combined = " ".join([title, abstract, methods])

    keywords = [
        "randomized controlled trial", "randomised controlled trial",
        "randomized trial", "randomised trial", "randomized", "randomised",
        "randomly", "random allocation", "double-blind", "double blind",
        "placebo-controlled", "placebo controlled", "placebo"
    ]
    return any(k in combined for k in keywords)


def is_review(art: Dict) -> bool:
    """
    Slightly broadened review detection: publication_types and title/abstract cues.
    """
    pts = []
    raw_pts = art.get("publication_types")
    if raw_pts:
        for pt in raw_pts:
            if isinstance(pt, str):
                pts.append(pt.lower())
            else:
                try:
                    pts.append(str(pt).lower())
                except Exception:
                    pass

    if any(("systematic review" in pt) or ("meta-analysis" in pt) for pt in pts):
        return True

    title = (art.get("title") or "").lower()
    abstract = (art.get("abstract") or "").lower()
    if "systematic review" in title or "meta-analysis" in title or "systematic review" in abstract or "meta-analysis" in abstract:
        return True

    return False


In [41]:
def get_evidence_bundle(claim_id: str) -> Dict:
    retrieved = RETRIEVAL_BY_CLAIM.get(claim_id, [])
    return {"rcts": [], "reviews": [], "retrieved": retrieved}


In [42]:
def build_articles_block(articles: List[Dict]) -> str:
    """Format multiple abstracts into a single block for the LLM prompt."""
    blocks = []
    for idx, art in enumerate(articles, start=1):
        lines = [
            f"Article {idx}:",
            f"PubMed ID: {art.get('pubmed_id', '')}",
            f"Evidence type: {art.get('_evidence_type', '')}",
            f"Year: {art.get('year', '')}",
            f"Title: {art.get('title', '')}",
            "Abstract:",
            art.get("abstract", "") or ""
        ]
        blocks.append("\n".join(lines))
    return "\n\n".join(blocks)


def build_toulmin_prompt(claim_text: str, articles: List[Dict]) -> str:
    articles_block = build_articles_block(articles)
    return TOULMIN_TEMPLATE.substitute(
        claim_text=claim_text,
        articles_block=articles_block
    )


In [43]:
def extract_toulmin_argument(articles: List[Dict], claim_text: str) -> Dict:
    if not articles:
        return {
            "stance": "irrelevant",
            "toulmin": {
                "claim": "",
                "data": [],
                "warrant": "",
                "backing": [],
                "qualifier": "",
                "rebuttals": []
            }
        }

    prompt = build_toulmin_prompt(claim_text, articles)

    response = client.chat.completions.create(
        model=OPENAI_CFG["model"],
        temperature=OPENAI_CFG.get("temperature", 0),
        response_format={"type": "json_object"},
        messages=[
            {"role": "system", "content": "You are a precise biomedical argument extraction assistant."},
            {"role": "user", "content": prompt}
        ]
    )

    content = response.choices[0].message.content
    try:
        parsed = json.loads(content)
    except json.JSONDecodeError:
        parsed = {
            "stance": "parse_error",
            "toulmin": {
                "claim": "",
                "data": [],
                "warrant": "",
                "backing": [],
                "qualifier": "",
                "rebuttals": [],
                "raw": content
            }
        }

    return parsed


In [44]:
def process_claim(claim: Dict):
    claim_id = claim["claim_id"]
    claim_text = claim["claim_text"]
    drug = claim["drug"]
    condition = claim.get("condition", "")
    outcome = claim.get("outcome", "")

    print(f"=== Processing claim {claim_id} ===")
    print(f"Drug: {drug}, Condition: {condition}, Outcome: {outcome}")
    print(f"Text: {claim_text}")

    bundle = get_evidence_bundle(claim_id)

    all_articles: List[Dict] = []
    for art in bundle["rcts"]:
        art["_evidence_type"] = "rct"
        all_articles.append(art)
    for art in bundle["reviews"]:
        art["_evidence_type"] = "review"
        all_articles.append(art)

    for art in bundle.get("retrieved", []):
        all_articles.append(art)

    print(f"Total articles to process with LLM: {len(all_articles)}")

    toulmin_arg = extract_toulmin_argument(all_articles, claim_text)

    sources = [
        {
            "pubmed_id": art.get("pubmed_id") or art.get("pmid"),
            "title": art.get("title"),
            "year": art.get("year"),
            "journal": art.get("journal"),
            "evidence_type": art.get("_evidence_type"),
            "abstract": art.get("abstract")
        }
        for art in all_articles
    ]

    record = {
        "claim_id": claim_id,
        "claim_text": claim_text,
        "drug": drug,
        "condition": condition,
        "outcome": outcome,
        "sources": sources,
        "stance": toulmin_arg.get("stance"),
        "toulmin": toulmin_arg.get("toulmin", {})
    }

    out_path = OUTPUT_DIR / f"{claim_id}_toulmin.jsonl"
    with out_path.open("w", encoding="utf-8") as f_out:
        f_out.write(json.dumps(record, ensure_ascii=False) + "")

    print(f"Saved Toulmin arguments for {claim_id} to {out_path}")


In [45]:
def run_pipeline():
    if not RETRIEVAL_RESULTS_PATH.exists():
        print(f"Missing retrieval results: {RETRIEVAL_RESULTS_PATH}")
        return

    with RETRIEVAL_RESULTS_PATH.open("r", encoding="utf-8") as f:
        lines = [ln.strip() for ln in f if ln.strip()]

    if not lines:
        print("No retrieval records found.")
        return

    for line in lines:
        rec = json.loads(line)
        claim_id = rec.get("claim_id", "retrieval")
        claim_text = rec.get("query", "")
        articles = rec.get("articles", [])

        print(f"=== Processing retrieval record {claim_id} ===")
        print(f"Query: {claim_text}")
        print(f"Total articles to process with LLM: {len(articles)}")

        toulmin_arg = extract_toulmin_argument(articles, claim_text)

        sources = [
            {
                "pubmed_id": art.get("pubmed_id") or art.get("pmid"),
                "title": art.get("title"),
                "year": art.get("year"),
                "journal": art.get("journal"),
                "evidence_type": art.get("_evidence_type"),
                "abstract": art.get("abstract")
            }
            for art in articles
        ]

        record = {
            "claim_id": claim_id,
            "claim_text": claim_text,
            "sources": sources,
            "stance": toulmin_arg.get("stance"),
            "toulmin": toulmin_arg.get("toulmin", {})
        }

        out_path = OUTPUT_DIR / f"{claim_id}_toulmin.jsonl"
        with out_path.open("w", encoding="utf-8") as f_out:
            f_out.write(json.dumps(record, ensure_ascii=False) + "")

        print(f"Saved Toulmin arguments for {claim_id} to {out_path}")


In [46]:
run_pipeline()

=== Processing retrieval record retrieval ===
Query: Semaglutide induces significant weight loss in adults with obesity.
Total articles to process with LLM: 15
Saved Toulmin arguments for retrieval to /Users/ftzavellos/Law_and_Tech/drug_explanations/Code/outputs/retrieval_toulmin.jsonl


## Cell for debugging

In [47]:
from pprint import pprint

def debug_evidence(drug, condition, outcome, max_show=5):
    print(f"Querying RCTs for: drug={drug}, condition={condition}, outcome={outcome}\n")
    q_rct = build_rct_query(drug, condition, outcome)
    print("Query for rct is:", q_rct)
    rct_candidates = fetch_articles(q_rct, max_results=PUBMED_CFG.get('max_results', 20))
    print("Fetched RCT candidates:", len(rct_candidates))
    for i, a in enumerate(rct_candidates[:max_show], start=1):
        print("\n--- Candidate", i, "---")
        print("pubmed_id:", a.get("pubmed_id"))
        print("title:", (a.get("title") or "")[:300])
        print("year (raw):", a.get("year"))
        try:
            print("year (norm):", normalize_year(a.get("year")))
        except Exception as e:
            print("normalize_year error:", e)
        print("publication_types:", a.get("publication_types"))
        print("keys:", list(a.keys()))
        abstract = a.get("abstract") or ""
        print("abstract snippet:", abstract[:300])
        pprint({k: a.get(k) for k in ("journal", "authors")})

    year_from = PUBMED_CFG["year_from"]
    filtered = filter_by_year(rct_candidates, year_from)
    print(f"\nAfter filter_by_year (>= {year_from}): {len(filtered)}")
    rcts = [a for a in filtered if is_rct(a)]
    print("is_rct matched:", len(rcts))
    fallback = [
        a for a in filtered
        if ("random" in (a.get("title") or "").lower() or "random" in (a.get("abstract") or "").lower())
    ]
    print("fallback 'random' matches:", len(fallback))
    return rct_candidates, filtered, rcts, fallback

# Example run using the first active claim
active = [c for c in CLAIMS if c.get("active", False)]
if active:
    c = active[0]
    print("Using claim:", c.get("claim_id"))
    debug_evidence(c["drug"], c.get("condition", ""), c.get("outcome", ""))
else:
    print("No active claims loaded; call debug_evidence(drug, condition, outcome) manually")

No active claims loaded; call debug_evidence(drug, condition, outcome) manually
